# 1. Import Libraries and Load Datasets

In [ ]:
import os
import pandas as pd
import configparser

In [ ]:
config = configparser.ConfigParser()
config.read(r'../config.ini')

validPath = False

datasetPaths = config.get('DataCleansing', 'DatasetPath').split(',')
for path in datasetPaths:
    try:
        print(f'Trying {path.strip()}...')
        os.chdir(path.strip())
        validPath = True
        print('Path found')
        break
    except FileNotFoundError:
        print(f'Path "{path.strip()}" not found')

if not validPath:
    raise RuntimeError('Could not find any valid dataset path')

# 2. Data Cleansing (Customer)

In [ ]:
customerChunk = pd.read_csv("training-data-customer-features_v1.0.csv", chunksize=200000)
ogCustomer = pd.concat([chunk for chunk in customerChunk])
ogCustomer.head()

In [ ]:
ogCustomer.shape

## A. Check for Missing Values

In [ ]:
ogCustomer.isnull().sum()

In [ ]:
percentageOfNa = round(ogCustomer.iloc[:,-1].isnull().sum() / len(ogCustomer) * 100, 2)
print(f'{percentageOfNa}% of the rows contain missing values. Since {percentageOfNa}% is very insignificant, we will drop the missing values')

In [ ]:
dropnaCustomer = ogCustomer.dropna()
dropnaCustomer.isnull().sum()

## B. Check for Duplicated Rows

Since our unique identifier for each row is a composite key comprised of `country_code` + `customer_id`, we will treat a row as a duplicate if its composite key is duplicated, even if the other columns are different

In [ ]:
print(dropnaCustomer.duplicated(subset=['country_code', 'customer_id']).sum())

As newer data records are always and nearer to the end of the table, we will use the last record of that customer and discard all other records from the same customer

In [ ]:
dropDuplicatesCustomer = dropnaCustomer.drop_duplicates(keep='last', subset=['country_code', 'customer_id'])
print(dropDuplicatesCustomer.duplicated(subset=['country_code', 'customer_id']).sum())

## C. Check Datatypes

In [ ]:
dropDuplicatesCustomer.info(memory_usage='deep')

### i. `country_code`

In [ ]:
dropDuplicatesCustomer['country_code'] = dropDuplicatesCustomer['country_code'].astype('category')

### ii. `first_order_datetime`

In [ ]:
dropDuplicatesCustomer.iloc[:,-1]

As we are unsure whether all of the dates are of the same format, we can check using regular expressions

In [ ]:
import re

pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"

def checkDatetimeFormat(datetimes):
    return all(re.match(pattern, dt) for dt in datetimes)

print(checkDatetimeFormat(dropDuplicatesCustomer.iloc[:,-1].values))


After we have ensured that they are all in the same format, we can convert them to `datetime` datatype

In [ ]:
dropDuplicatesCustomer['first_order_datetime'] = pd.to_datetime(dropDuplicatesCustomer['first_order_datetime'], format='%Y-%m-%d %H:%M:%S')

In [ ]:
dropDuplicatesCustomer.iloc[:,-1].unique()[:5]

### iii. `mobile_verified`

As SQL does not have a boolean datatype with `T` or `F` values but instead a `BIT` with `1` and `0` values, we will convert the `mobile_verified` column to `int` datatype by converting `True` to `1` and `False` to `0`

In [ ]:
dropDuplicatesCustomer.iloc[:,2].unique()

In [ ]:
dropDuplicatesCustomer.iloc[:,2] = dropDuplicatesCustomer.iloc[:,2].astype('uint8')

In [ ]:
dropDuplicatesCustomer.iloc[:,2].unique()

### iv. `num_orders_last_50days`, `num_cancelled_orders_last_50days`, `num_refund_orders_last_50days`, `num_associated_customers`

In [ ]:
print(f'Min and max value of num_orders_last_50days: {dropDuplicatesCustomer["num_orders_last_50days"].min()} {dropDuplicatesCustomer["num_orders_last_50days"].max()}')
print(f'Min and max value of num_cancelled_orders_last_50days: {dropDuplicatesCustomer["num_cancelled_orders_last_50days"].min()} {dropDuplicatesCustomer["num_cancelled_orders_last_50days"].max()}')
print(f'Min and max value of num_refund_orders_last_50days: {dropDuplicatesCustomer["num_refund_orders_last_50days"].min()} {dropDuplicatesCustomer["num_refund_orders_last_50days"].max()}')
print(f'Min and max value of num_associated_customers: {dropDuplicatesCustomer["num_associated_customers"].min()} {dropDuplicatesCustomer["num_associated_customers"].max()}')

In [ ]:
dropDuplicatesCustomer[['num_orders_last_50days', 'num_cancelled_orders_last_50days', 'num_refund_orders_last_50days']] = dropDuplicatesCustomer[['num_orders_last_50days', 'num_cancelled_orders_last_50days', 'num_refund_orders_last_50days']].astype('int16')
dropDuplicatesCustomer['num_associated_customers'] = dropDuplicatesCustomer['num_associated_customers'].astype('int8')

In [ ]:
dropDuplicatesCustomer.info(memory_usage='deep')

From the memory usage report shown above, we have reduced the dataset memory usage from 300+ MB to 120+ MB

## D. Exporting Cleaned Dataset

In [ ]:
customer = dropDuplicatesCustomer
ogCustomerLength = len(ogCustomer)
customerLength = len(customer)

print(ogCustomer.shape)
print(customer.shape)
print(f'We removed {ogCustomerLength - customerLength} rows which is {((ogCustomerLength - customerLength) / ogCustomerLength * 100):.2f}% of the total number of rows in the dataset')

Although 41.4% is quite a significant amount of data cleansed from our dataset, we still have more than 1.2 million to work with for our model training which is not too little

In [ ]:
customer.to_csv('final_customer.csv', index=False)

# 3. Data Cleansing (Order)

In [ ]:
orderChunk = pd.read_csv("training-data-order-features_v1.0.csv", chunksize=200000)
ogOrder = pd.concat([chunk for chunk in orderChunk])
ogOrder.head()

In [ ]:
ogOrder.shape

## A. Check for Missing Values

In [ ]:
ogOrder.isnull().sum()

In [ ]:
dropnaOrder = ogOrder.dropna()
dropnaOrder.isnull().sum()

## B. Check for Duplicated Rows

In [ ]:
print(dropnaOrder.duplicated().sum())

In [ ]:
dropDuplicatesOrder = dropnaOrder.drop_duplicates(keep='last')
print(dropDuplicatesOrder.duplicated().sum())

## C. Check Datatypes

In [ ]:
dropDuplicatesOrder.info(memory_usage='deep')

### i. `country_code`, `payment_method` and `collect_type`

In [ ]:
print(dropDuplicatesOrder['country_code'].unique())
print(dropDuplicatesOrder['payment_method'].unique())
print(dropDuplicatesOrder['collect_type'].unique())

In [ ]:
dropDuplicatesOrder[['collect_type', 'payment_method', 'country_code']] = dropDuplicatesOrder[['collect_type', 'payment_method', 'country_code']].astype('category')

### ii. `num_items_ordered`

In [ ]:
# check if all the floats can be converted to int without any data loss
(dropDuplicatesOrder['num_items_ordered'] % 1 == 0).all()

In [ ]:
print(f'max: {dropDuplicatesOrder["num_items_ordered"].max()}')
print(f'min: {dropDuplicatesOrder["num_items_ordered"].min()}')

As the floating points are not useful to the data, we can remove and convert them into a smaller sized `int16` datatype. 

In [ ]:
dropDuplicatesOrder["num_items_ordered"] = dropDuplicatesOrder["num_items_ordered"].astype('int16')

### iii. `order_date`

In [ ]:
dropDuplicatesOrder.iloc[:,-1].unique()[:5]

As the `order_date` is in `object` datatype, we will convert its format from `YYYY/MM/DD` TO `YYYY-MM-DD` to ensure consistency in the format of the dates

In [ ]:
dropDuplicatesOrder['order_date'] = dropDuplicatesOrder['order_date'].str.replace('/', '-', regex=False)
dropDuplicatesOrder['order_date'] = pd.to_datetime(dropDuplicatesOrder['order_date'], format='%Y-%m-%d')

In [ ]:
dropDuplicatesOrder.iloc[:,-1].unique()[:5]

In [ ]:
dropDuplicatesOrder.info(memory_usage='deep')

From the memory usage report shown above, we have reduced the dataset memory usage from 780+ MB to 220+ MB

## D. Exporting Cleaned Dataset

In [ ]:
order = dropDuplicatesOrder
ogOrderLength = len(ogOrder)
orderLength = len(order)

print(ogOrder.shape)
print(order.shape)
print(f'We removed {ogOrderLength - orderLength} rows which is {((ogOrderLength - orderLength) / ogOrderLength * 100):.2f}% of the total number of rows in the dataset')

In [ ]:
order.to_csv('final_order.csv', index=False)

# 4. Data Cleansing (Labels)

In [ ]:
labelsChunk = pd.read_csv("training-labels_v1.0.csv", chunksize=200000)
ogLabels = pd.concat([chunk for chunk in labelsChunk])
ogLabels.head()

In [ ]:
ogLabels.shape

## A. Checking for Missing Values

In [ ]:
ogLabels.isnull().sum()

## B. Checking for Duplicated Rows

In [ ]:
ogLabels.duplicated(subset=['country_code', 'order_id', 'customer_id']).sum()

In [ ]:
dropDuplicatesLabels = ogLabels.drop_duplicates(subset=['country_code', 'order_id', 'customer_id'], keep='last')
dropDuplicatesLabels.duplicated().sum()

## C. Check Datatypes

In [ ]:
dropDuplicatesLabels.info(memory_usage='deep')

### i. `country_code`

In [ ]:
dropDuplicatesLabels['country_code'] = dropDuplicatesLabels['country_code'].astype('category')

### ii. `is_fraud`

As the values of `is_fraud` can only be either 0 or 1 and is always positive, we can convert the datatype into an unsigned integer `uint8`

In [ ]:
dropDuplicatesLabels['is_fraud'] = dropDuplicatesLabels['is_fraud'].astype('uint8')

In [ ]:
dropDuplicatesLabels.info(memory_usage='deep')

From the memory usage report shown above, we have reduced the dataset memory usage from 440+ MB to 300+ MB

## D. Exporting Cleaned Dataset

In [ ]:
labels = dropDuplicatesLabels
ogLabelsLength = len(ogLabels)
labelsLength = len(labels)

print(ogLabels.shape)
print(labels.shape)
print(f'We removed {ogLabelsLength - labelsLength} rows which is {((ogLabelsLength - labelsLength) / ogLabelsLength * 100):.2f}% of the total number of rows in the dataset')

In [ ]:
labels.to_csv('final_labels.csv', index=False)